# Avaliacao de Comites de Classificacao
## Imports e funções

In [1]:
import traceback
import pandas as pd
import numpy as np
from numpy import mean
from numpy import std
from sklearn import metrics
from sklearn.metrics import confusion_matrix, f1_score

from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict
from sklearn.naive_bayes import GaussianNB, MultinomialNB, ComplementNB

from sklearn.model_selection import ParameterGrid

def raca_para_especie(raca):
    if raca in ['basset_hound', 'saint_bernard']:
        return 'dog'
    elif raca in ['Birman', 'Persian']:
        return 'cat'
    else:
        return raca  # fallback

### Para Bases com PCA que geram valores negativos
from sklearn.preprocessing import minmax_scale

def separar_dataset(df, scale=False):
    X = df.iloc[:, :-1]
    if scale:
        X = minmax_scale(X)
    y = df.iloc[:, -1]
    return X, y




## Lendo lista de arquivos a processar

In [2]:
datafiles = pd.read_csv('dataset_list.csv',encoding='utf-8')

datafiles.head(12)

,key,filename
0,hogfeat_128_16_4_9_pca,../Aula11/hogfeat_128_16_4_9_pca.csv.gz
1,hogfeat_256_64_2_18,../Aula11/hogfeat_256_64_2_18.csv.gz
2,hogfeat_256_64_2_9,../Aula11/hogfeat_256_64_2_9.csv.gz
3,hogfeat_128_16_4_9,../Aula11/hogfeat_128_16_4_9.csv.gz
4,hogfeat_256_32_2_9_pca,../Aula11/hogfeat_256_32_2_9_pca.csv.gz
5,hogfeat_128_16_2_9_pca,../Aula11/hogfeat_128_16_2_9_pca.csv.gz
6,hogfeat_128_32_2_9,../Aula11/hogfeat_128_32_2_9.csv.gz
7,lbpfeat_256_6_48,../Aula11/lbpfeat_256_6_48.csv.gz
8,lbpfeat_256_12_96,../Aula11/lbpfeat_256_12_96.csv.gz
9,hogfeat_256_32_2_9,../Aula11/hogfeat_256_32_2_9.csv.gz


## Lendo Dataframes e ajustando dados

In [3]:

dfs = {}
shapes = []
last_cols = []

for metadata in datafiles.itertuples():
    # Imprime o arquivo que está sendo lido
    #print(f"Lendo arquivo: {metadata.key}")
    # Carrega o DataFrame
    df = pd.read_csv(metadata.filename)
    
    # Aplica a função raca_para_especie na coluna raca
    if 'raca' in df.columns:
        df['especie'] = df['raca'].apply(raca_para_especie)
        df = df.drop('raca', axis=1)
    
    # Elimina a coluna nome_arquivo se existir
    if 'nome_arquivo' in df.columns:
        df = df.drop('nome_arquivo', axis=1)
    
    dfs[metadata.key] = df
    shapes.append(df.shape)
    # Pega as últimas duas colunas
    last_cols.append(df.columns[-1:].tolist())

# Adiciona as colunas shape e last_two_columns ao datafiles
datafiles['shape'] = shapes
datafiles['last_column'] = last_cols

datafiles
    

,key,filename,shape,last_column
0,hogfeat_128_16_4_9_pca,../Aula11/hogfeat_128_16_4_9_pca.csv.gz,"(800, 104)",[especie]
1,hogfeat_256_64_2_18,../Aula11/hogfeat_256_64_2_18.csv.gz,"(800, 649)",[especie]
2,hogfeat_256_64_2_9,../Aula11/hogfeat_256_64_2_9.csv.gz,"(800, 325)",[especie]
3,hogfeat_128_16_4_9,../Aula11/hogfeat_128_16_4_9.csv.gz,"(800, 3601)",[especie]
4,hogfeat_256_32_2_9_pca,../Aula11/hogfeat_256_32_2_9_pca.csv.gz,"(800, 94)",[especie]
5,hogfeat_128_16_2_9_pca,../Aula11/hogfeat_128_16_2_9_pca.csv.gz,"(800, 118)",[especie]
6,hogfeat_128_32_2_9,../Aula11/hogfeat_128_32_2_9.csv.gz,"(800, 325)",[especie]
7,lbpfeat_256_6_48,../Aula11/lbpfeat_256_6_48.csv.gz,"(800, 51)",[especie]
8,lbpfeat_256_12_96,../Aula11/lbpfeat_256_12_96.csv.gz,"(800, 99)",[especie]
9,hogfeat_256_32_2_9,../Aula11/hogfeat_256_32_2_9.csv.gz,"(800, 1765)",[especie]


## Aplicando as configurações nas 12 bases de dados

In [4]:
# Aplicando configurações Bagging em todas as bases de dados
param_grid = {
    'estimator': [
        DecisionTreeClassifier(criterion="gini", max_depth=8),
        KNeighborsClassifier(n_neighbors=4, metric='euclidean'), 
        GaussianNB(),
        MLPClassifier( hidden_layer_sizes=(100,50), activation='logistic', solver='sgd', max_iter=2000, learning_rate_init=0.1),
    ],
    'n_estimators': [10, 20, 30],
    'training': ["holdout", "crossvalidation"]
}

# Estrutura para armazenar resultados
resultados_bagging = []

param_combinations = list(ParameterGrid(param_grid))
print(f"Total de combinações a testar: {len(param_combinations)}")

for j, metadata in enumerate(datafiles.itertuples(), start=1):
    print("#" * 50)
    print(f"Dataset {j}: {metadata.key}")
    
    # Obtém o dataset correspondente do dicionário dfs
    df = dfs[metadata.key]
    X, y = separar_dataset(df, False)
        
    # Testando cada uma das configurações
    for i, params in enumerate(param_combinations):
        print(f"Testando configuração {i+1}/{len(param_combinations)}: {params}")

        # for parametro, valor in params.items():
        #     print(f"\t{parametro}: {valor}")

        algorithm = params['estimator']          # O classificador base
        num_estimators = params['n_estimators']    # Número de estimadores
        training_type = params['training']       # Tipo de treinamento ('holdout' ou 'crossvalidation')

        try:
            # Criando o modelo com os parâmetros específicos
            f1 = 0.0
            f1_std = 0.0
            confusao = np.array([])
            
            bgclassifier = BaggingClassifier(estimator=algorithm,
                        n_estimators=num_estimators, max_features=1.0, max_samples=1.0)

            if training_type == "holdout":
                # Treinamento e avaliação usando holdout
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
                
                # Capturando warnings durante o treinamento
                bgclassifier.fit(X_train, y_train)
                    
                
                y_pred = bgclassifier.predict(X_test)
                f1 = f1_score(y_test, y_pred, average='weighted')
                f1_std = 0.0
                confusao = confusion_matrix(y_test, y_pred)
                
                print(f"\t✓ Sucesso - F1: {f1:.4f}")
                    
            elif training_type == "crossvalidation":
                # Treinamento e avaliação usando crossvalidation
                kf = KFold(n_splits=10, random_state=42, shuffle=True)
                
                scores = cross_val_score(bgclassifier, X, y, scoring='f1_weighted', cv=kf)
                y_pred = cross_val_predict(bgclassifier, X, y, cv=kf)
                    
                confusao = confusion_matrix(y, y_pred)
                f1 = scores.mean()
                f1_std = scores.std()
                
                print(f"\t✓ Sucesso - F1: {f1:.4f} ({f1_std:.4f})")
            else:
                print(f"\tTraining type {training} nao suportado")
                continue
            
            # Armazena resultados no DataFrame
            resultado = {
                'dataset': metadata.key,
                'config_rank': i + 1,
                'params': params,
                'training_type': training_type,
                'f1_score': f1,
                'f1_std': f1_std,
                'confusion_matrix': confusao,
                'model': 'BaggingClassifier'
            }
            resultados_bagging.append(resultado)
            
        except Exception as e:
            traceback.print_exc()
            print(f"\t✗ ERRO: {type(e).__name__}")
            continue
        
        print("-" * 50)

# Converte para DataFrame
df_resultados_bagging = pd.DataFrame(resultados_bagging)
                
df_resultados_bagging['shape'] = df_resultados_bagging['dataset'].apply(lambda x: datafiles.loc[datafiles['key'] == x, 'shape'].values[0])



Total de combinações a testar: 24
##################################################
Dataset 1: hogfeat_128_16_4_9_pca
Testando configuração 1/24: {'estimator': DecisionTreeClassifier(max_depth=8), 'n_estimators': 10, 'training': 'holdout'}
	✓ Sucesso - F1: 0.7838
--------------------------------------------------
Testando configuração 2/24: {'estimator': DecisionTreeClassifier(max_depth=8), 'n_estimators': 10, 'training': 'crossvalidation'}
	✓ Sucesso - F1: 0.7168 (0.0452)
--------------------------------------------------
Testando configuração 3/24: {'estimator': DecisionTreeClassifier(max_depth=8), 'n_estimators': 20, 'training': 'holdout'}
	✓ Sucesso - F1: 0.7591
--------------------------------------------------
Testando configuração 4/24: {'estimator': DecisionTreeClassifier(max_depth=8), 'n_estimators': 20, 'training': 'crossvalidation'}
	✓ Sucesso - F1: 0.7317 (0.0407)
--------------------------------------------------
Testando configuração 5/24: {'estimator': DecisionTreeClass

KeyboardInterrupt: 

## Salvando resultados em CSV e Joblib

In [54]:
# Salvando resultados MLP
import joblib
joblib.dump(df_resultados_mlp, 'mlp_avaliacao_resultados.joblib')

df_csv_mlp = df_resultados_mlp.drop(['confusion_matrix', 'model'], axis=1)
df_csv_mlp.to_csv('mlp_avaliacao_resultados.csv', index=False)

print(f"Resultados MLP salvos:")
print(f"- mlp_avaliacao_resultados.csv: {len(df_csv_mlp)} registros")
print(f"- mlp_avaliacao_resultados.joblib: dados completos com modelos")

Resultados MLP salvos:
- mlp_avaliacao_resultados.csv: 240 registros
- mlp_avaliacao_resultados.joblib: dados completos com modelos


### Amostra do DataFrame de Resultados

In [55]:
# Visualiza os resultados MLP
print(f"Total de combinações processadas: {len(df_resultados_mlp)}")
print(f"Colunas disponíveis: {df_resultados_mlp.columns.tolist()}")
print(f"Datasets processados: {df_resultados_mlp['dataset'].nunique()}")
print(f"Configurações por dataset: {df_resultados_mlp['config_rank'].nunique()}")

# Análise dos melhores resultados por dataset
melhores_por_dataset = df_resultados_mlp.loc[df_resultados_mlp.groupby('dataset')['f1_score'].idxmax()]
print(f"\nMelhores F1 scores por dataset:")
for _, row in melhores_por_dataset.iterrows():
    print(f"  {row['dataset']}: {row['f1_score']:.4f} (Config {row['config_rank']}, {row['training_type']})")

df_resultados_mlp


Total de combinações processadas: 240
Colunas disponíveis: ['dataset', 'config_rank', 'params', 'training_type', 'f1_score', 'f1_std', 'confusion_matrix', 'model', 'shape']
Datasets processados: 12
Configurações por dataset: 10

Melhores F1 scores por dataset:
  hogfeat_128_16_2_9: 0.8131 (Config 10, holdout)
  hogfeat_128_16_2_9_pca: 0.7876 (Config 6, holdout)
  hogfeat_128_16_4_9: 0.8048 (Config 8, holdout)
  hogfeat_128_16_4_9_pca: 0.7764 (Config 10, crossvalidation)
  hogfeat_128_32_2_9: 0.7653 (Config 6, crossvalidation)
  hogfeat_256_32_2_9: 0.7926 (Config 1, crossvalidation)
  hogfeat_256_32_2_9_pca: 0.8030 (Config 10, crossvalidation)
  hogfeat_256_64_2_18: 0.8001 (Config 8, crossvalidation)
  hogfeat_256_64_2_9: 0.7788 (Config 6, crossvalidation)
  lbpfeat_256_12_96: 0.6884 (Config 2, holdout)
  lbpfeat_256_24_192: 0.6967 (Config 2, holdout)
  lbpfeat_256_6_48: 0.6634 (Config 5, crossvalidation)


,dataset,config_rank,params,training_type,f1_score,f1_std,confusion_matrix,model,shape
0,hogfeat_128_16_4_9_pca,1,"{'activation': 'tanh', 'hidden_layer_sizes': (...",holdout,0.763210,0.000000,"[[82, 24], [33, 101]]","MLPClassifier(activation='tanh', hidden_layer_...","(800, 104)"
1,hogfeat_128_16_4_9_pca,1,"{'activation': 'tanh', 'hidden_layer_sizes': (...",crossvalidation,0.746158,0.039768,"[[287, 113], [90, 310]]","MLPClassifier(activation='tanh', hidden_layer_...","(800, 104)"
2,hogfeat_128_16_4_9_pca,2,"{'activation': 'tanh', 'hidden_layer_sizes': 1...",holdout,0.754799,0.000000,"[[80, 26], [33, 101]]","MLPClassifier(activation='tanh', hidden_layer_...","(800, 104)"
3,hogfeat_128_16_4_9_pca,2,"{'activation': 'tanh', 'hidden_layer_sizes': 1...",crossvalidation,0.770159,0.025076,"[[307, 93], [91, 309]]","MLPClassifier(activation='tanh', hidden_layer_...","(800, 104)"
4,hogfeat_128_16_4_9_pca,3,"{'activation': 'tanh', 'hidden_layer_sizes': 1...",holdout,0.754799,0.000000,"[[80, 26], [33, 101]]","MLPClassifier(activation='tanh', hidden_layer_...","(800, 104)"
...,...,...,...,...,...,...,...,...,...
235,lbpfeat_256_24_192,8,"{'activation': 'tanh', 'hidden_layer_sizes': 1...",crossvalidation,0.320970,0.108902,"[[123, 277], [152, 248]]","MLPClassifier(activation='tanh', hidden_layer_...","(800, 195)"
236,lbpfeat_256_24_192,9,"{'activation': 'logistic', 'hidden_layer_sizes...",holdout,0.400089,0.000000,"[[0, 106], [0, 134]]","MLPClassifier(activation='logistic', hidden_la...","(800, 195)"
237,lbpfeat_256_24_192,9,"{'activation': 'logistic', 'hidden_layer_sizes...",crossvalidation,0.336109,0.076863,"[[0, 400], [0, 400]]","MLPClassifier(activation='logistic', hidden_la...","(800, 195)"
238,lbpfeat_256_24_192,10,"{'activation': 'logistic', 'hidden_layer_sizes...",holdout,0.270617,0.000000,"[[106, 0], [134, 0]]","MLPClassifier(activation='logistic', hidden_la...","(800, 195)"
